In [20]:
# Create three separate Matplotlib charts (no subplots) and then stack them into one image.
# We will avoid setting explicit colors to follow the constraints. We'll rely on default colors.
# The combined graphic will be saved as PNG, SVG, and PDF for easy inclusion in papers.

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

# Projekt-Root, eine Ebene raus
ROOT = Path.cwd().parent
ROOT

WindowsPath('c:/Users/User/Code/VSCProjects/Master_Thesis/multi-period-forecasting')

In [21]:
# ---------------- Config ----------------
# Projekt-Root wie im Rest deiner Notebooks:
OUT_DIR = ROOT / "docs" / "assets" / "images" / "evaluation_protocols"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Speichere nach:", OUT_DIR.resolve())

Speichere nach: C:\Users\User\Code\VSCProjects\Master_Thesis\multi-period-forecasting\docs\assets\images\evaluation_protocols


In [22]:

N = 40   # Gesamtlänge (Zeitindex)
H = 7    # Holdout / Horizon
ROLL_STARTS = [18, 20, 22, 24, 26, 28]  # Rolling-Origin Cutoffs
K = 5    # blocked k-fold

# ---- einheitliche Geometrie ----
FIG_W_IN    = 10.0    # Breite in Inch
ROW_H_IN    = 0.55    # Höhe pro Zeile in Inch (steuert optische Balkendicke)
FIG_PAD_IN  = 0.50    # Top/Bottom-Padding in Inch
BAR_H       = 0.50    # barh-Höhe (gleich in allen Plots)

# Farben (farbfehlsicher & kontrastreich)
TRAIN_COLOR = "#5672CD"  # blau
TEST_COLOR  = "#E69F00"  # orange




In [23]:
# ---------- Utility: schöne Legende außerhalb ----------
from matplotlib.patches import Patch
LEG_HANDLES = [Patch(facecolor=TRAIN_COLOR, label="Train"),
               Patch(facecolor=TEST_COLOR,  label="Test")]

def _style_axis(ax, title, rows, xlim=(0, N), yticks=None, yticklabels=None, legend_loc="center left"):
    ax.set_title(title)
    ax.set_xlim(*xlim)
    ax.set_ylim(-0.5, rows-0.5)  # konsistente Ränder
    if yticks is None:
        ax.set_yticks([])
    else:
        ax.set_yticks(yticks)
        if yticklabels is not None:
            ax.set_yticklabels(yticklabels)
    ax.set_xlabel("Zeit (Index)")
    # Platz für die Legende rechts
    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width*0.85, box.height])
    ax.legend(handles=LEG_HANDLES, loc=legend_loc, bbox_to_anchor=(1.01, 0.5), frameon=False)
    ax.grid(axis="x", linestyle="--", alpha=0.35)

# ---------- 1) Fixed Holdout ----------
def save_fixed_holdout_png(path_png):
    rows = 1
    fig_h_in = FIG_PAD_IN + rows * ROW_H_IN
    fig = plt.figure(figsize=(FIG_W_IN, fig_h_in), dpi=200)
    ax = plt.gca()

    train_len = N - H
    ax.barh(y=[0], width=[train_len], left=[0], height=BAR_H, color=TRAIN_COLOR)
    ax.barh(y=[0], width=[H],        left=[train_len], height=BAR_H, color=TEST_COLOR)

    _style_axis(
        ax,
        title="Chronologischer Fixed-Holdout-Split",
        rows=rows,
        yticks=[0],
        yticklabels=["Split 1"],          # <— einheitlich
        legend_loc="center left"
    )
    fig.savefig(path_png, bbox_inches="tight")
    plt.close(fig)

# ---------- 2) Rolling-Origin / Walk-Forward ----------
def save_rolling_origin_png(path_png):
    rows = len(ROLL_STARTS)
    fig_h_in = FIG_PAD_IN + rows * ROW_H_IN
    fig = plt.figure(figsize=(FIG_W_IN, fig_h_in), dpi=200)
    ax = plt.gca()

    yrows = list(range(rows))[::-1]
    for i, t in enumerate(ROLL_STARTS):
        ax.barh(y=[yrows[i]], width=[t], left=[0], height=BAR_H, color=TRAIN_COLOR)
        test_w = max(0, min(H, N - t))
        if test_w > 0:
            ax.barh(y=[yrows[i]], width=[test_w], left=[t], height=BAR_H, color=TEST_COLOR)

    _style_axis(
        ax,
        title="Rolling-Origin / Walk-Forward (wachsendes Trainingsfenster)",
        rows=rows,
        yticks=yrows,
        yticklabels=[f"Split {i+1}" for i in range(rows)],  # <— einheitlich
        legend_loc="center left"
    )
    fig.savefig(path_png, bbox_inches="tight")
    plt.close(fig)

# ---------- 3) Zeitreihen-CV (blocked k-fold) ----------
def save_blocked_kfold_png(path_png):
    rows = K
    fig_h_in = FIG_PAD_IN + rows * ROW_H_IN
    fig = plt.figure(figsize=(FIG_W_IN, fig_h_in), dpi=200)
    ax = plt.gca()

    fold_size = N // K
    yrows = list(range(rows))[::-1]

    for i in range(K):
        test_left = i * fold_size
        test_w = (N - test_left) if i == K - 1 else fold_size

        if test_left > 0:
            ax.barh(y=[yrows[i]], width=[test_left], left=[0], height=BAR_H, color=TRAIN_COLOR)
        ax.barh(y=[yrows[i]], width=[test_w], left=[test_left], height=BAR_H, color=TEST_COLOR)

        after_left = test_left + test_w
        after_w = max(0, N - after_left)
        if after_w > 0:
            ax.barh(y=[yrows[i]], width=[after_w], left=[after_left], height=BAR_H, color=TRAIN_COLOR)

    _style_axis(
        ax,
        title="Zeitreihen-Cross-Validation (blocked k-fold)",
        rows=rows,
        yticks=yrows,
        yticklabels=[f"Split {i+1}" for i in range(rows)],  # <— einheitlich
        legend_loc="center left"
    )
    fig.savefig(path_png, bbox_inches="tight")
    plt.close(fig)


In [24]:
# ---------- Rendern & Speichern ----------
p_fixed    = OUT_DIR / "ts_eval_fixed_holdout.png"
p_rolling  = OUT_DIR / "ts_eval_rolling_origin.png"
p_kfold    = OUT_DIR / "ts_eval_blocked_kfold.png"

save_fixed_holdout_png(p_fixed)
save_rolling_origin_png(p_rolling)
save_blocked_kfold_png(p_kfold)

print("Gespeichert:")
print(" -", p_fixed.name)
print(" -", p_rolling.name)
print(" -", p_kfold.name)


Gespeichert:
 - ts_eval_fixed_holdout.png
 - ts_eval_rolling_origin.png
 - ts_eval_blocked_kfold.png


In [25]:
# ---------- Kombiniertes PNG (optional) ----------
try:
    from PIL import Image
    imgs = [Image.open(p) for p in [p_fixed, p_rolling, p_kfold]]
    width = max(im.width for im in imgs)
    gutter = 24
    total_height = sum(im.height for im in imgs) + gutter*(len(imgs)-1)
    combined = Image.new("RGB", (width, total_height), "white")
    y = 0
    for i, im in enumerate(imgs):
        combined.paste(im, (0, y))
        y += im.height + (gutter if i < len(imgs)-1 else 0)
    p_combined = OUT_DIR / "ts_evaluation_protocols_combined.png"
    combined.save(p_combined)
    print(" -", p_combined.name)
except Exception as e:
    print("Kombiniertes PNG übersprungen (Pillow fehlt?):", e)


 - ts_evaluation_protocols_combined.png
